# Missing Value Treatment

Missing data is an unavoidable reality in Data Science. A user forgets to fill out a field on a form, a sensor loses power for five minutes, or a database merge fails to find a match. 

When dealing with missing data, you generally have two main choices: **Delete it** or **Fill it (Impute it)**. The right choice depends entirely on how much data is missing and why it is missing.

Let's set up a new Python sandbox with a deliberately incomplete dataset.

In [1]:
import pandas as pd
import numpy as np

# Create a dataset with missing values (np.nan)
data = {
    'customer_id': [1, 2, 3, 4, 5, 6],
    'age': [25, 32, np.nan, 28, 45, np.nan], 
    'income': [50000, 65000, 120000, np.nan, 80000, 55000],
    'city': ['New York', 'London', 'London', np.nan, 'Paris', 'New York']
}

df = pd.DataFrame(data)

print("--- Original Messy Data ---")
display(df)
print("\n--- Missing Value Count ---")
print(df.isnull().sum())

--- Original Messy Data ---


,customer_id,age,income,city
0,1,25.0,50000.0,New York
1,2,32.0,65000.0,London
2,3,NaN,120000.0,London
3,4,28.0,NaN,NaN
4,5,45.0,80000.0,Paris
5,6,NaN,55000.0,New York



--- Missing Value Count ---
customer_id    0
age            2
income         1
city           1
dtype: int64


# 1. Strategy 1: Deletion (Dropping Data)
The easiest way to handle missing data is to simply throw it away. You can either throw away the entire row, or the entire column.

* **Dropping Rows (`axis=0`)**: Good if only a tiny percentage of your dataset (e.g., 1%) has missing values. 
* **Dropping Columns (`axis=1`)**: Good if a specific column is overwhelmingly empty (e.g., 80% missing). If a column is mostly blank, it holds no predictive power anyway.

In [2]:
# 1. Drop any ROW that has at least one missing value
df_dropped_rows = df.dropna(axis=0)
print("--- Data after Dropping Rows ---")
display(df_dropped_rows)

# 2. Drop any COLUMN that has at least one missing value
df_dropped_cols = df.dropna(axis=1)
print("\n--- Data after Dropping Columns ---")
display(df_dropped_cols)

--- Data after Dropping Rows ---


,customer_id,age,income,city
0,1,25.0,50000.0,New York
1,2,32.0,65000.0,London
4,5,45.0,80000.0,Paris



--- Data after Dropping Columns ---


,customer_id
0,1
1,2
2,3
3,4
4,5
5,6


*(Notice the danger of dropping rows: We started with 6 customers and ended up with only 3! We threw away a lot of perfectly good data just because one piece was missing. This is why we usually prefer **Imputation**.)*

# 2. Strategy 2: Simple Imputation (Pandas `fillna`)
Instead of throwing data away, we can make an educated guess and fill in the blanks. This is called **Imputation**. 

Using Pandas' `.fillna()` method, we can fill missing values with summary statistics.
* **Mean (Average)**: Good for normally distributed numbers without extreme outliers.
* **Median (Middle value)**: Better for numbers that have extreme outliers (like income).
* **Mode (Most frequent)**: The *only* simple option for text/categorical data!

In [3]:
# Create a copy so we don't mess up our original dataframe
df_imputed = df.copy()

# Fill missing 'age' with the MEAN (average) age
mean_age = df_imputed['age'].mean()
df_imputed['age'] = df_imputed['age'].fillna(mean_age)

# Fill missing 'income' with the MEDIAN income
median_income = df_imputed['income'].median()
df_imputed['income'] = df_imputed['income'].fillna(median_income)

# Fill missing 'city' with the MODE (most frequent city)
# Note: mode() returns a Series, so we grab the first item [0]
mode_city = df_imputed['city'].mode()[0]
df_imputed['city'] = df_imputed['city'].fillna(mode_city)

print("--- Data after Simple Imputation ---")
display(df_imputed)

--- Data after Simple Imputation ---


,customer_id,age,income,city
0,1,25.0,50000.0,New York
1,2,32.0,65000.0,London
2,3,32.5,120000.0,London
3,4,28.0,65000.0,London
4,5,45.0,80000.0,Paris
5,6,32.5,55000.0,New York


# 3. Strategy 3: Machine Learning Imputation (Scikit-Learn)
While using Pandas `.fillna()` is great for quick analysis, it is actually considered a bad practice when building production Machine Learning models. 

Instead, we use **Scikit-Learn's `SimpleImputer`**. It does the exact same math as Pandas, but it can be seamlessly integrated into an automated Machine Learning "Pipeline" (which we will learn about later in Lesson 10).

In [4]:
from sklearn.impute import SimpleImputer

# 1. Create the Imputer objects
# We define our strategy: "Fill numbers with the median, fill text with the most frequent"
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# 2. Apply the imputers to the original messy data
df_sklearn = df.copy()

# Fit and transform the numerical columns
df_sklearn[['age', 'income']] = num_imputer.fit_transform(df_sklearn[['age', 'income']])

# Fit and transform the categorical column
df_sklearn[['city']] = cat_imputer.fit_transform(df_sklearn[['city']])

print("--- Data after Scikit-Learn Imputation ---")
display(df_sklearn)

--- Data after Scikit-Learn Imputation ---


,customer_id,age,income,city
0,1,25.0,50000.0,New York
1,2,32.0,65000.0,London
2,3,30.0,120000.0,London
3,4,28.0,65000.0,London
4,5,45.0,80000.0,Paris
5,6,30.0,55000.0,New York


# 4. Strategy 4: Forward Fill and Backward Fill
If you are dealing with **Time Series Data** (like daily stock prices or weather temperatures), filling a missing Tuesday temperature with the "Average Yearly Temperature" makes no sense.

Instead, you use `ffill` (Forward Fill) to take Monday's temperature and drag it forward into Tuesday. Or you use `bfill` (Backward Fill) to take Wednesday's temperature and drag it backward.

In [5]:
# A quick time-series example
time_data = pd.DataFrame({
    'date': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri'],
    'temperature': [70, np.nan, np.nan, 76, 78]
})

print("--- Original Time Series ---")
display(time_data)

# Forward Fill: Propagate the last valid observation forward
print("\n--- After Forward Fill (ffill) ---")
display(time_data.ffill())

--- Original Time Series ---


,date,temperature
0,Mon,70.0
1,Tue,NaN
2,Wed,NaN
3,Thu,76.0
4,Fri,78.0



--- After Forward Fill (ffill) ---


,date,temperature
0,Mon,70.0
1,Tue,70.0
2,Wed,70.0
3,Thu,76.0
4,Fri,78.0


## Real-World Use Case or Analogy:
Think of Missing Value Treatment like **Restoring an Antique Brick Wall**:

* **The Problem**: A 100-year-old brick wall is missing a few bricks. If you leave the holes, the building inspector (the Machine Learning model) will fail the building and shut it down.
* **Deletion (Dropping rows)**: You take a sledgehammer and knock down the entire section of the wall just to get rid of the hole. You fixed the hole, but now your house is significantly smaller.
* **Mean Imputation**: You measure the average size and color of every brick in the wall, manufacture a perfectly average, generic brick, and shove it into the hole. It works, but if you look closely, it lacks nuance.
* **Mode Imputation**: You look around and notice that 80% of the bricks in this specific wall are painted Red. You buy a Red brick and put it in the hole. (Used for text/categories).
* **Forward Fill**: This is a sequential pattern. You notice the brick directly to the left of the hole has a beautiful ivy pattern carved into it. You carve that exact same ivy pattern into a new brick and push it into the hole, assuming the pattern was meant to continue.

---